# Spotify Tracks EDA\nExploratory analysis for genre classification and recommendation.

In [ ]:
import pandas as pd\nimport seaborn as sns\nimport matplotlib.pyplot as plt\nfrom pathlib import Path\n\nsns.set_style('whitegrid')\ndata_path = Path('../data/raw')\ncsv_files = sorted(data_path.glob('*.csv'))\nif not csv_files:\n    raise FileNotFoundError('Place Spotify CSV in data/raw/')\ndf = pd.read_csv(csv_files[0])\ndf.head()

In [ ]:
# Genre distribution\nplt.figure(figsize=(14, 5))\ntop_genres = df['track_genre'].value_counts().head(20)\nsns.barplot(x=top_genres.index, y=top_genres.values, palette='viridis')\nplt.xticks(rotation=75)\nplt.title('Top 20 Genre Distribution')\nplt.tight_layout()\nplt.show()

In [ ]:
# Audio feature distributions\naudio_features = [\n    'danceability','energy','loudness','speechiness','acousticness',\n    'instrumentalness','liveness','valence','tempo'\n]\ndf[audio_features].hist(figsize=(15, 10), bins=30)\nplt.tight_layout()\nplt.show()

In [ ]:
# Correlation heatmap\nplt.figure(figsize=(10, 8))\ncorr = df[audio_features].corr(numeric_only=True)\nsns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')\nplt.title('Audio Feature Correlation Matrix')\nplt.tight_layout()\nplt.show()

In [ ]:
# Feature importance using RandomForest\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import LabelEncoder\n\nfeature_cols = [c for c in audio_features + ['popularity','duration_ms','explicit','key','mode','time_signature'] if c in df.columns]\ntmp = df[feature_cols + ['track_genre']].dropna().copy()\nif 'explicit' in tmp.columns:\n    tmp['explicit'] = tmp['explicit'].map({True:1, False:0, 'True':1, 'False':0}).fillna(0)\n\nX = tmp[feature_cols]\nle = LabelEncoder()\ny = le.fit_transform(tmp['track_genre'])\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)\nrf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)\nrf.fit(X_train, y_train)\nimportance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)\nplt.figure(figsize=(10, 5))\nsns.barplot(x=importance.index, y=importance.values, palette='magma')\nplt.xticks(rotation=60)\nplt.title('Feature Importance (RandomForest)')\nplt.tight_layout()\nplt.show()